# Phase 1 Kaggle Ablation - Qwen 0.5B SFT

This notebook is the Kaggle-safe ablation runner for the next step after the 1000-step canary. It uses the production CLI, but keeps the model download, YAML generation, checkpoint lookup, and test-loss evaluation visible.

Recommended order:

1. Run the current 1000-step GSM8K-only canary.
2. Run this three-arm SFT ablation: GSM8K-only, MATH-only, combined GSM8K+MATH.
3. Compare GSM8K and MATH test loss for each arm.
4. Add generation exact-answer evaluation after the loss-only picture is stable.

DPO is intentionally not in this notebook. DPO needs a preference dataset or preference-generation pipeline; this notebook answers whether the SFT data mixture is worth scaling first.

## Step 1 - GPU and runtime check

Use Kaggle GPU, preferably T4. Do not use P100 if PyTorch warns that sm_60 is unsupported.

In [ ]:
!nvidia-smi

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import torch
    print('torch:', torch.__version__)
    print('cuda available:', torch.cuda.is_available())
    print('device count:', torch.cuda.device_count())
    if torch.cuda.is_available():
        for idx in range(torch.cuda.device_count()):
            print(idx, torch.cuda.get_device_name(idx), torch.cuda.get_device_capability(idx))
except Exception as exc:
    print('torch check failed:', repr(exc))

## Step 2 - Repo setup and import path

The CLI runs in a new Python process, so it needs `PYTHONPATH=/kaggle/working/finpost/src`, not just notebook `sys.path`.

In [ ]:
REPO_DIR = Path('/kaggle/working/finpost')
SRC_DIR = REPO_DIR / 'src'

%cd /kaggle/working
!test -d finpost || git clone https://github.com/shannan-liu1/finpost.git
%cd /kaggle/working/finpost
!pip install -q -e .

os.environ['PYTHONPATH'] = str(SRC_DIR)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

for name in list(sys.modules):
    if name == 'finpost' or name.startswith('finpost.'):
        del sys.modules[name]

import finpost
print('finpost loaded from:', finpost.__file__)

## Step 3 - Settings

Defaults are conservative for Kaggle T4. If your 1000-step CLI canary is stable at `max_seq_len=1024`, you can raise `MAX_SEQ_LEN` back to 1024. For first ablations, `512` is safer and cheaper.

In [ ]:
from datetime import datetime

MODEL_REPO_ID = 'Qwen/Qwen2.5-0.5B'
MODEL_DIR = Path('/kaggle/working/models/qwen2_5_0_5b')

ABLATION_STEPS = 1000
MAX_SEQ_LEN = 512
DTYPE = 'float32'
LR = 1.0e-5
PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
VAL_SPLIT_PCT = 5.0
SEED = 42

# Start with one arm if you want a short smoke pass. Then run all three.
RUN_ARMS = ['gsm8k_only', 'math_only', 'combined']

# For quick eval during notebook iteration. Set to None for full test splits.
EVAL_MAX_EXAMPLES = 500

# Unique suffix prevents checkpoint collisions when rerunning the same arm.
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')

ARMS = {
    'gsm8k_only': ['gsm8k'],
    'math_only': ['math'],
    'combined': ['gsm8k', 'math'],
}

print('arms:', RUN_ARMS)
print('steps:', ABLATION_STEPS)
print('max_seq_len:', MAX_SEQ_LEN)
print('effective batch:', PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS)
print('run tag:', RUN_TAG)

## Step 4 - Optional Hugging Face token

Qwen does not require a token, but setting `HF_TOKEN` can reduce rate-limit noise. For gated models later, this becomes required.

In [ ]:
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret('HF_TOKEN')
except Exception as exc:
    print('No Kaggle HF_TOKEN secret loaded:', repr(exc))

if HF_TOKEN:
    from huggingface_hub import login
    os.environ['HF_TOKEN'] = HF_TOKEN
    login(token=HF_TOKEN)
    print('HF token loaded')
else:
    print('Proceeding without HF token')

## Step 5 - Download model once, then run locally

This avoids the partial `model.safetensors` download stalls that happen when `from_pretrained` fetches directly during model load.

In [ ]:
!pip -q install hf_transfer
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

from huggingface_hub import snapshot_download

snapshot_download(
    repo_id=MODEL_REPO_ID,
    local_dir=str(MODEL_DIR),
    token=HF_TOKEN,
    allow_patterns=[
        '*.json',
        '*.safetensors',
        'tokenizer.*',
        'vocab.json',
        'merges.txt',
    ],
)

required = ['config.json']
missing = [name for name in required if not (MODEL_DIR / name).exists()]
if missing:
    raise RuntimeError(f'Model download incomplete; missing {missing}')

print('model dir:', MODEL_DIR)
print('files:', sorted(p.name for p in MODEL_DIR.iterdir())[:20])

## Step 6 - Choose the CLI GPU

If Kaggle gives two T4s, this picks the GPU with the most free memory for the subprocess. The CLI still uses a single GPU.

In [ ]:
def pick_cli_gpu() -> str:
    try:
        raw = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.free', '--format=csv,noheader,nounits'],
            text=True,
        )
        free = [int(line.strip()) for line in raw.splitlines() if line.strip()]
        if not free:
            return '0'
        return str(max(range(len(free)), key=lambda idx: free[idx]))
    except Exception as exc:
        print('Could not query GPU memory:', repr(exc))
        return '0'

CLI_GPU = pick_cli_gpu()
print('CLI GPU index:', CLI_GPU)

CLI_ENV = os.environ.copy()
CLI_ENV['PYTHONPATH'] = str(SRC_DIR)
CLI_ENV['WANDB_MODE'] = 'offline'
CLI_ENV['CUDA_VISIBLE_DEVICES'] = CLI_GPU

subprocess.run(
    [sys.executable, '-c', 'import finpost; print(finpost.__file__)'],
    cwd=str(REPO_DIR),
    env=CLI_ENV,
    check=True,
)

## Step 7 - Write one YAML per ablation arm

The only intended difference across arms is the data mixture. Keep model, LR, batch, dtype, sequence length, seed, and training steps fixed. A timestamp run tag is included so reruns do not collide with old step directories.

In [ ]:
import yaml

CONFIG_DIR = Path('/kaggle/working/phase1_ablation_configs')
SAVE_ROOT = Path('/kaggle/working/results/checkpoints/phase1_ablation')
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

warmup_steps = max(1, min(ABLATION_STEPS // 10, ABLATION_STEPS - 1))
val_every = max(50, min(250, ABLATION_STEPS // 4))

def make_arm_config(arm_name: str, sources: list[str]) -> dict:
    run_name = f'qwen-ablate-{arm_name}-{ABLATION_STEPS}s-seq{MAX_SEQ_LEN}-{RUN_TAG}'
    return {
        'model': {
            'base_model_id': str(MODEL_DIR),
            'dtype': DTYPE,
            'use_safetensors': True,
        },
        'data': {
            'sources': sources,
            'val_split_pct': VAL_SPLIT_PCT,
            'seed': SEED,
        },
        'training': {
            'max_steps': ABLATION_STEPS,
            'warmup_steps': warmup_steps,
            'lr': LR,
            'weight_decay': WEIGHT_DECAY,
            'grad_accum_steps': GRAD_ACCUM_STEPS,
            'grad_clip': GRAD_CLIP,
            'val_every_n_steps': val_every,
            'checkpoint_every_n_steps': ABLATION_STEPS,
            'per_device_batch_size': PER_DEVICE_BATCH_SIZE,
        },
        'packing': {
            'max_seq_len': MAX_SEQ_LEN,
            'isolate_documents': True,
        },
        'logging': {
            'wandb_project': 'finpost-phase1',
            'run_name': run_name,
        },
        'checkpointing': {
            'save_dir': str(SAVE_ROOT / run_name),
            'retention_last_n': 1,
            'resume_from': None,
        },
    }

ARM_CONFIGS = {}
for arm_name, sources in ARMS.items():
    cfg = make_arm_config(arm_name, sources)
    path = CONFIG_DIR / f'{arm_name}.yaml'
    with path.open('w', encoding='utf-8') as fp:
        yaml.safe_dump(cfg, fp, sort_keys=False)
    ARM_CONFIGS[arm_name] = {'path': path, 'config': cfg}
    print(arm_name, '->', path)
    print('  sources:', sources)
    print('  save_dir:', cfg['checkpointing']['save_dir'])

## Step 8 - Clear notebook GPU memory before CLI runs

This notebook should not have a model on GPU before the CLI starts. If `nvidia-smi` still shows several GB for the notebook process, restart the kernel and run only setup cells plus the CLI cells.

In [ ]:
import gc

for name in ['model', 'optimizer', 'scheduler', 'out', 'outputs', 'loss', 'batch']:
    if name in globals():
        del globals()[name]

gc.collect()
if 'torch' in globals() and torch.cuda.is_available():
    torch.cuda.empty_cache()

!nvidia-smi

## Step 9 - Run the ablation arms

This can take a while. For a smoke pass, set `RUN_ARMS = ['gsm8k_only']` in Step 3 and rerun from there.

In [ ]:
def run_arm(arm_name: str) -> None:
    info = ARM_CONFIGS[arm_name]
    cfg_path = info['path']
    print(f'\n=== Running {arm_name} ===')
    print('config:', cfg_path)
    cmd = [
        sys.executable,
        '-m',
        'finpost.training.train',
        '--config',
        str(cfg_path),
        '--device',
        'cuda',
        '--max-steps',
        str(ABLATION_STEPS),
    ]
    subprocess.run(cmd, cwd=str(REPO_DIR), env=CLI_ENV, check=True)

for arm_name in RUN_ARMS:
    if arm_name not in ARMS:
        raise ValueError(f'Unknown arm: {arm_name}')
    run_arm(arm_name)

## Step 10 - Locate checkpoints

The repo checkpoint format is `step-XXXXXXXX/model.safetensors` plus `state.pt`. It is not a Hugging Face `save_pretrained` folder; reload it with `finpost.training.checkpoint.load_checkpoint`.

In [ ]:
def latest_checkpoint(save_dir: str | Path) -> Path:
    save_dir = Path(save_dir)
    candidates = sorted(
        [p for p in save_dir.glob('step-*') if p.is_dir() and not p.name.endswith('.tmp')],
        key=lambda p: int(p.name.split('-')[-1]),
    )
    if not candidates:
        raise FileNotFoundError(f'No step-* checkpoints under {save_dir}')
    return candidates[-1]

CHECKPOINTS = {}
for arm_name, info in ARM_CONFIGS.items():
    save_dir = info['config']['checkpointing']['save_dir']
    if Path(save_dir).exists():
        try:
            CHECKPOINTS[arm_name] = latest_checkpoint(save_dir)
            print(arm_name, '->', CHECKPOINTS[arm_name])
        except FileNotFoundError as exc:
            print(arm_name, 'missing:', exc)
    else:
        print(arm_name, 'save_dir missing:', save_dir)

## Step 11 - Loss-only test evaluation

This evaluates gold-response cross entropy on held-out test splits. It does not measure generated final-answer accuracy. Use it to compare arms cheaply and consistently.

In [ ]:
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from finpost.data.gsm8k import load_gsm8k
from finpost.data.math_dataset import load_math
from finpost.training.checkpoint import load_checkpoint
from finpost.training.dataset import (
    PackingCollator,
    TokenizedSFTExample,
    serialize_prompt,
    serialize_response,
)
from finpost.training.masking import IGNORE_INDEX

class TestSFTDataset(Dataset):
    def __init__(self, examples, tokenizer):
        self.examples = examples
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        prompt_ids = self.tokenizer(serialize_prompt(ex.prompt), add_special_tokens=False)['input_ids']
        response_ids = self.tokenizer(serialize_response(ex.response), add_special_tokens=False)['input_ids']
        ids = torch.tensor(prompt_ids + response_ids, dtype=torch.long)
        return TokenizedSFTExample(
            input_ids=ids,
            prompt_length=len(prompt_ids),
            source=ex.source,
            example_id=ex.id,
        )

def load_test_examples(source: str):
    if source == 'gsm8k':
        examples = load_gsm8k(split='test')
    elif source == 'math':
        examples = load_math(split='test')
    else:
        raise ValueError(source)
    if EVAL_MAX_EXAMPLES is not None:
        examples = examples[:EVAL_MAX_EXAMPLES]
    return examples

def ce_sum_and_count(logits: torch.Tensor, labels: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    flat_logits = shift_logits.view(-1, shift_logits.size(-1))
    flat_labels = shift_labels.view(-1)
    valid = flat_labels != IGNORE_INDEX
    if valid.sum() == 0:
        return torch.tensor(0.0, device=logits.device), valid.sum()
    loss_sum = F.cross_entropy(flat_logits[valid], flat_labels[valid], reduction='sum')
    return loss_sum, valid.sum()

def load_model_from_trainer_checkpoint(ckpt_dir: Path):
    state = load_checkpoint(ckpt_dir)
    tok = AutoTokenizer.from_pretrained(state.config.model.base_model_id, local_files_only=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    dtype = getattr(torch, state.config.model.dtype)
    mdl = AutoModelForCausalLM.from_pretrained(
        state.config.model.base_model_id,
        dtype=dtype,
        use_safetensors=state.config.model.use_safetensors,
        local_files_only=True,
    ).to('cuda')
    missing, unexpected = mdl.load_state_dict(state.model_state_dict, strict=False)
    print('loaded step:', state.step, 'missing:', len(missing), 'unexpected:', len(unexpected))
    mdl.eval()
    return mdl, tok, state.config

def evaluate_test_loss(ckpt_dir: Path, test_source: str) -> dict:
    model, tokenizer, cfg = load_model_from_trainer_checkpoint(ckpt_dir)
    examples = load_test_examples(test_source)
    collator = PackingCollator(
        max_seq_len=cfg.packing.max_seq_len,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        isolate_documents=cfg.packing.isolate_documents,
    )
    loader = DataLoader(
        TestSFTDataset(examples, tokenizer),
        batch_size=cfg.training.per_device_batch_size,
        shuffle=False,
        collate_fn=collator,
    )

    total_nll = 0.0
    total_tokens = 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to('cuda')
            labels = batch['labels'].to('cuda')
            attention_mask = batch['attention_mask'].to('cuda').bool()
            position_ids = batch['position_ids'].to('cuda')
            out = model(input_ids=input_ids, attention_mask=attention_mask, position_ids=position_ids)
            loss_sum, count = ce_sum_and_count(out.logits, labels)
            total_nll += float(loss_sum.detach().float().item())
            total_tokens += int(count.item())

    avg_loss = total_nll / total_tokens if total_tokens else float('nan')
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return {
        'checkpoint': str(ckpt_dir),
        'test_source': test_source,
        'examples': len(examples),
        'tokens': total_tokens,
        'loss': avg_loss,
        'perplexity': float(torch.exp(torch.tensor(avg_loss)).item()) if total_tokens else float('nan'),
    }

## Step 12 - Compare arms on GSM8K and MATH test loss

For the clean ablation, every completed arm is evaluated on both test sets. The useful table is row = training arm, columns = test source losses.

In [ ]:
import pandas as pd

EVAL_SOURCES = ['gsm8k', 'math']
rows = []

for arm_name in RUN_ARMS:
    if arm_name not in CHECKPOINTS:
        print('Skipping missing checkpoint:', arm_name)
        continue
    for source in EVAL_SOURCES:
        print(f'Evaluating {arm_name} on {source} test')
        row = evaluate_test_loss(CHECKPOINTS[arm_name], source)
        row['arm'] = arm_name
        row['trained_on'] = '+'.join(ARMS[arm_name])
        rows.append(row)

results = pd.DataFrame(rows)
results = results[['arm', 'trained_on', 'test_source', 'examples', 'tokens', 'loss', 'perplexity', 'checkpoint']]
results_path = Path('/kaggle/working/phase1_ablation_test_loss.csv')
results.to_csv(results_path, index=False)
print('saved:', results_path)
results

## Step 13 - Plot the comparison

In [ ]:
import matplotlib.pyplot as plt

if 'results' not in globals() or results.empty:
    raise RuntimeError('Run Step 12 first')

pivot = results.pivot(index='arm', columns='test_source', values='loss')
ax = pivot.plot(kind='bar', figsize=(9, 4), rot=0)
ax.set_ylabel('masked cross-entropy test loss')
ax.set_title(f'Qwen 0.5B SFT ablation ({ABLATION_STEPS} steps, seq {MAX_SEQ_LEN})')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plot_path = Path('/kaggle/working/phase1_ablation_test_loss.png')
plt.savefig(plot_path, dpi=160)
print('saved:', plot_path)
plt.show()

## Step 14 - Zip checkpoints and outputs for download

In [ ]:
import shutil
from IPython.display import FileLink

archive_base = '/kaggle/working/phase1_ablation_outputs'
zip_path = shutil.make_archive(archive_base, 'zip', '/kaggle/working/results/checkpoints/phase1_ablation')
print('checkpoint archive:', zip_path)
FileLink(zip_path)

In [ ]:
from IPython.display import FileLink, display

for path in [
    '/kaggle/working/phase1_ablation_test_loss.csv',
    '/kaggle/working/phase1_ablation_test_loss.png',
]:
    if Path(path).exists():
        display(FileLink(path))
    else:
        print('missing:', path)

## Optional - exact-answer eval comes next

Loss-only eval is the right first comparison because it is stable and fast. The next layer is generation-based exact answer: prompt only, generate, parse the final answer, compare to `Example.final_answer`. Keep that separate because decoding settings can dominate the result.